# PaLM — Scaling Language Modeling with Pathways (2022)

_Papers · Transformers & LLMs_

**A 540-billion-parameter dense Transformer, trained across thousands of TPU chips, whose reasoning skills jump sharply with scale.**

---

This notebook is a practice scaffold for the **PaLM — Scaling Language Modeling with Pathways (2022)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# (1) WORKED EXAMPLE: SwiGLU gate  Swish(a) * b  vs plain ReLU(a).
#     This is the REAL SwiGLU formula from the paper (Sec 2), on toy scalars.
# ---------------------------------------------------------------------------
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def swish(z):   return z * sigmoid(z)          # Swish(z) = z * sigmoid(z)
def swiglu(a, b): return swish(a) * b          # SwiGLU = Swish(xW) . xV
def relu(z):    return np.maximum(0.0, z)

for a, b in [(2.0, 3.0), (-1.0, 3.0), (1.0, 4.0)]:
    print("a=%5.1f b=%5.1f | SwiGLU=%.5f | ReLU(a)=%.5f" % (
          a, b, swiglu(a, b), relu(a)))
# a=  2.0 b=  3.0 | SwiGLU=5.28478 | ReLU(a)=2.00000
# a= -1.0 b=  3.0 | SwiGLU=-0.80682 | ReLU(a)=0.00000
# a=  1.0 b=  4.0 | SwiGLU=2.92423 | ReLU(a)=1.00000
# SwiGLU depends on BOTH branches and leaks a small signed value where ReLU
# hard-clips to 0. These match the worked example in the lesson.

# ---------------------------------------------------------------------------
# (2) SYNTHETIC "EMERGENT JUMP": accuracy vs log10(model params). The paper
#     reports many tasks stay near chance, then jump sharply with scale
#     (Sec 6.2). We MAKE UP a smooth step to show the SHAPE -- these are NOT
#     PaLM's numbers and not any real benchmark.
# ---------------------------------------------------------------------------
log_params = np.array([8.0, 9.0, 10.0, 11.0, 12.0])   # ~1e8 .. 1e12 params
threshold, sharpness, chance, ceiling = 11.0, 4.0, 0.05, 0.85
acc = chance + (ceiling - chance) * sigmoid(sharpness * (log_params - threshold))
print("\nsynthetic emergent curve (ILLUSTRATION ONLY, not PaLM numbers):")
for lp, a in zip(log_params, acc):
    print("  log10(params)=%.1f  acc=%.3f" % (lp, a))
# synthetic emergent curve (ILLUSTRATION ONLY, not PaLM numbers):
#   log10(params)=8.0  acc=0.050
#   log10(params)=9.0  acc=0.050
#   log10(params)=10.0 acc=0.064
#   log10(params)=11.0 acc=0.450
#   log10(params)=12.0 acc=0.836
# Flat-then-jump: the shape the paper calls "discontinuous improvements from
# scale" (Sec 6.2). Round, made-up numbers -- NOT a PaLM benchmark.

## Visualize the data & results

_Two illustrations of PaLM ideas on toy data: (1) how does the SwiGLU gate Swish(a)*b differ from plain ReLU(a) across inputs? (2) what does a 'discontinuous improvement from scale' curve look like? (All numbers ours, not the paper's.)_

In [ ]:
import numpy as np

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def swish(z):   return z * sigmoid(z)

# LEFT panel: SwiGLU gate Swish(a)*b (b=3) vs ReLU(a) -- the REAL formula.
a = np.array([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0])
b = 3.0
swiglu = swish(a) * b
relu   = np.maximum(0.0, a)
print("a       SwiGLU(*b=3)   ReLU(a)")
for ai, si, ri in zip(a, swiglu, relu):
    print("  %5.1f   %8.4f   %6.4f" % (ai, si, ri))
# a       SwiGLU(*b=3)   ReLU(a)
#   -3.0    -0.4268   0.0000
#   -2.0    -0.7152   0.0000
#   -1.0    -0.8068   0.0000
#    0.0     0.0000   0.0000
#    1.0     2.1932   1.0000
#    2.0     5.2848   2.0000
#    3.0     8.5732   3.0000

# RIGHT panel: SYNTHETIC emergent jump -- ROUND, MADE-UP numbers, NOT PaLM's.
log_params = np.array([8.0, 9.0, 10.0, 11.0, 12.0])
threshold, sharpness, chance, ceiling = 11.0, 4.0, 0.05, 0.85
acc = chance + (ceiling - chance) * sigmoid(sharpness * (log_params - threshold))
print("\nsynthetic emergent curve (ILLUSTRATION ONLY):")
for lp, ac in zip(log_params, acc):
    print("  log10(params)=%.1f  acc=%.3f" % (lp, ac))
# synthetic emergent curve (ILLUSTRATION ONLY):
#   log10(params)=8.0  acc=0.050
#   log10(params)=9.0  acc=0.050
#   log10(params)=10.0 acc=0.064
#   log10(params)=11.0 acc=0.450
#   log10(params)=12.0 acc=0.836
# Flat near chance, then a sharp jump: the SHAPE the paper calls 'discontinuous
# improvements from scale' (Sec 6.2). NOT a real PaLM benchmark score.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recompute the SwiGLU gate. For a hidden unit with gate-branch value $a = 1.0$ and
            value-branch value $b = 4.0$, compute the SwiGLU output $\text{Swish}(a)\cdot b$, and compare it
            to the plain ReLU output $\text{ReLU}(a)$. (Use $e \approx 2.71828$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sigmoid of the gate branch: $\sigma(1.0) = 1/(1 + e^{-1.0}) = 1/(1 + 0.36788) = 0.73106$. — _Swish needs $\sigma(a)$; $e^{-1} \approx 0.36788$._
- Swish: $\text{Swish}(1.0) = 1.0 \times 0.73106 = 0.73106$. — _$\text{Swish}(z) = z\,\sigma(z)$._
- Gate the value branch: SwiGLU $= 0.73106 \times 4.0 = 2.92424$. ReLU $= \max(0, 1.0) = 1.0$. — _SwiGLU multiplies the smooth gate by the second branch $b$; ReLU ignores $b$ and just clips $a$._

**Answer:** SwiGLU output $\approx 0.731 \times 4.0 = \mathbf{2.924}$; ReLU output $= \mathbf{1.0}$.
                 The gate ($\approx 0.731$) lets most of the value branch through and the result depends on
                 both branches, whereas ReLU passes only the (clipped) gate input and discards the
                 value branch entirely. This is the gating behaviour the CODEVIZ plots.

</details>

**Problem 2.** Count the savings in a parallel block. Compare the standard serial block
            $y = x + \text{MLP}(\text{LayerNorm}(x + \text{Attention}(\text{LayerNorm}(x))))$ with PaLM's
            parallel block $y = x + \text{MLP}(\text{LayerNorm}(x)) + \text{Attention}(\text{LayerNorm}(x))$.
            (a) How many LayerNorm calls per block in each? (b) Which two input projections can be fused in the
            parallel form, and why does that need them to act on the same vector?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count LayerNorms in the serial block: one before Attention on $x$, one before the MLP on $(x + \text{Attention}(...))$ — that is 2 distinct calls on 2 distinct inputs. — _Each sublayer normalizes its own input, and those inputs differ._
- Count LayerNorms in the parallel block: both branches read $\text{LayerNorm}(x)$ — that is 1 call, shared. — _In the parallel form both sublayers consume the identical normalized vector._
- Identify the fusable projections: the MLP's input matrix and attention's query/key/value projection both multiply $\text{LayerNorm}(x)$, so they can be concatenated into one matmul. — _Two matrices applied to the same input vector are equivalent to one stacked matrix applied once — a single larger matmul, which is faster on a TPU._

**Answer:** (a) Serial block: 2 LayerNorm calls (different inputs). Parallel block: 1
                 LayerNorm call, shared by both branches. (b) The MLP's first-layer projection and attention's
                 QKV projection both act on the same vector $\text{LayerNorm}(x)$, so they can be fused
                 into a single large matrix multiply. The fusion is only possible because both branches
                 read the same input — that is exactly what the parallel form arranges. One LayerNorm plus one
                 fused matmul is the source of the "roughly 15% faster training at large scales" (§2).

</details>

**Problem 3.** Ablation — break the parallel block. Suppose you revert PaLM 540B to serial blocks
            (the standard form) and change nothing else. Per the paper's own statements, predict the effect on
            (a) training speed and (b) final quality at the 540B scale, and explain the mechanism for each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Speed: reverting to serial restores two LayerNorm calls and unfused projections, so you lose the "roughly 15% faster training speed at large scales" (§2). — _The parallel form's speed came precisely from the single LayerNorm and the fused input matmul; undo it and the work returns._
- Quality: the paper reports the parallel form's quality cost is small at 8B and shrinks at 62B (negligible by large scale), so reverting buys little or no quality at 540B. — _The empirical finding is that the parallel-vs-serial quality gap closes as scale grows._
- Conclude the trade: at 540B, serial is slower with no meaningful quality gain — so PaLM keeps parallel. — _That is the paper's justification for the choice: speed for negligible cost at scale._

**Answer:** (a) Training gets slower — you give back the ~15% speedup, because the two
                 efficiency sources (single shared LayerNorm, fused input projection) are gone. (b) Final
                 quality at 540B is essentially unchanged: the paper found the parallel form's quality
                 cost is small at 8B and negligible by large scale. So the ablation is a clear loss — slower
                 training for no quality payoff — which is exactly why PaLM adopts parallel layers (§2). This
                 illustrates the general theme: several PaLM choices trade a tiny small-scale cost for a real
                 large-scale benefit.

</details>